# Kalshi Bot v2 — full pipeline

Thin notebook. All logic lives in `kalshi_v2/`. Cells here just
drive the lifecycle (build clients → start bot → inspect → stop)
and print results inline.

**Prereq.** `~/.kalshi/credentials.env` must contain:
```
KALSHI_PROD_KEY_ID=...
KALSHI_PROD_PRIVATE_KEY_PATH=~/.kalshi/prod_private_key.pem
```

**Pipeline.** Coinbase BTC spot → Kalshi WS orderbooks → empirical
bank fair value (drift-removed, vol+kurt matched) → HRDNN P-robust
filter (16 bootstrapped measures) → Lipschitz-clamped sizing →
risk preflight → place / record.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from datetime import datetime, timezone

from kalshi_v2.config import CFG
from kalshi_v2.client import KalshiClient
from kalshi_v2 import data as v2data
from kalshi_v2.data import (fetch_historical_minutes, add_rv_features,
                              SPOT, BOOKS, TRACKED, WS_STATE, BOT_STATE)
from kalshi_v2.model import build_empirical_bank
from kalshi_v2.robust import build_ambiguity_set, SIZER
from kalshi_v2.strategy import scan_signals
from kalshi_v2.paper_db import open_trades, settled_trades
from kalshi_v2.risk import RISK_BLOCKS, get_live_balance
from kalshi_v2.portfolio import (paper_portfolio_metrics,
                                   live_portfolio_metrics, portfolio_metrics)
from kalshi_v2.main import (start_bot, stop_bot, status, kill_switch,
                              enable_live, disable_live, cancel_all_live_orders)

print('imports OK')
print(f'mode:          {CFG["mode"]}')
print(f'live_enabled:  {CFG["live_enabled"]}')
print(f'robust_enabled: {CFG["robust_enabled"]}')
print(f'sfm_enabled:   {CFG["sfm_enabled"]}')

## 2. Build clients

Two clients: `kalshi_md` for read-only market data (no auth required
for public endpoints) and `kalshi_live` for authed actions (orders,
balance, WS handshake).

In [ ]:
kalshi_md   = KalshiClient(env='prod')
kalshi_live = KalshiClient(env='prod')

print(f'md client:    base={kalshi_md.base_url}, signed={kalshi_md.private_key is not None}')
print(f'live client:  base={kalshi_live.base_url}, signed={kalshi_live.private_key is not None}')
print(f'live key_id:  {kalshi_live.key_id[:8] + "..." if kalshi_live.key_id else None}')

if kalshi_live.private_key is None:
    print('\n  ⚠ live client unsigned — bot will run paper-only, no WS auth')

In [ ]:
# Quick sanity check: hit a public endpoint
try:
    ev = kalshi_md.get_events(series_ticker='KXBTC', status='open', limit=3)
    print(f'public REST OK — {len(ev.get("events", []))} BTC events open')
    for e in ev.get('events', [])[:3]:
        print(f'  {e.get("event_ticker")}')
except Exception as e:
    print(f'REST sanity failed: {e}')

if kalshi_live.private_key is not None:
    try:
        bal = kalshi_live.get_balance()
        print(f'\nlive balance: ${float(bal.get("balance", 0))/100:.2f}')
    except Exception as e:
        print(f'\nbalance fetch failed: {e}')

## 3. Show config

In [ ]:
for k in sorted(CFG.keys()):
    v = CFG[k]
    print(f'  {k:30s} {v}')

## 4. Fetch BTC history + features

90 days of BTC 1-min bars from Coinbase. This populates the input to
`build_empirical_bank` and `build_ambiguity_set`. Takes 30–60 s.

In [ ]:
btc_1m = add_rv_features(fetch_historical_minutes(days_back=90))
print(f'btc_1m: {len(btc_1m):,} bars')
print(f'  range:   {btc_1m["time"].min()}  →  {btc_1m["time"].max()}')
print(f'  spot:    ${btc_1m["close"].iloc[-1]:,.0f}')
print(f'  rv_60m last:  {btc_1m["rv_60m"].iloc[-1]:.4f} (annualized)')
btc_1m.tail(3)

## 5. Build empirical bank — preview before bot starts

Diagnostic: same call `start_bot` will make. Confirms drift removal
and conditioning features are sane.

In [ ]:
bank = build_empirical_bank(btc_1m, horizon_min=60, n_samples=5000, demean=True)
R = bank['log_returns']
import numpy as np
print(f'  n samples:     {bank["n"]:,}')
print(f'  R mean (post-demean): {R.mean():+.6f}')
print(f'  R std:         {R.std():.6f}')
print(f'  R quantiles:   1%={np.quantile(R, 0.01):+.4f}, 50%={np.quantile(R, 0.50):+.4f}, 99%={np.quantile(R, 0.99):+.4f}')
print(f'  v_mean:        {bank["v_mean"]:.4f}')
print(f'  k_mean:        {bank["k_mean"]:+.3f}')

## 6. Build ambiguity set — preview

16 bootstrapped empirical banks. The robust filter rejects a signal
unless **every** measure agrees the trade has positive post-fee edge.
Spread of v_mean across measures shows the filter has real ambiguity
to test against (vs a near-zero spread, which would mean it's a noop).

In [ ]:
amb = build_ambiguity_set(btc_1m, horizon_min=60,
                            n_bootstrap=CFG['robust_n_bootstrap'])
v_means = [m['v_mean'] for m in amb]
k_means = [m['k_mean'] for m in amb]
print(f'  measures:        {len(amb)}')
print(f'  v_mean range:   [{min(v_means):.4f}, {max(v_means):.4f}]  (spread {max(v_means)-min(v_means):.4f})')
print(f'  k_mean range:   [{min(k_means):+.3f}, {max(k_means):+.3f}]  (spread {max(k_means)-min(k_means):.3f})')
if max(v_means) - min(v_means) < 0.01:
    print('  ⚠ low spread — bootstrap may not be giving ambiguity (degenerate ambiguity set)')
else:
    print('  ✓ measures vary — filter has something to test against')

## 7. Start the bot

Spins up 4 daemon threads:
- `spot_poller` — Coinbase BTC spot every 2 s
- `ws_listener` — Kalshi WS orderbook stream (REST fallback on 401/403)
- `event_tracker` — finds nearest BTC event in TTL window every 60 s
- `decision`     — settle → manage → scan → execute every `decision_interval_sec`

Idempotent: re-running `start_bot` while running prints a warning and
no-ops. Pass `btc_1m=btc_1m, refresh_btc_1m=False` to skip the 90-day
refetch (we already have it).

In [ ]:
start_bot(kalshi_md, kalshi_live, btc_1m=btc_1m, refresh_btc_1m=False)

## 8. Live status

Re-run this cell any time to see thread health, WS state, current
tracked event, recent log lines.

In [ ]:
status(last_n_log_lines=20)

## 9. Manual signal scan (diagnostic)

Calls the same `scan_signals` the decision worker calls, but inline
so you can see the edge distribution and which markets passed the
robust filter.

In [ ]:
from kalshi_v2.main import _EMPIRICAL_BANK, _AMBIGUITY_SET

sigs = scan_signals(empirical_bank=_EMPIRICAL_BANK,
                      ambiguity_set=_AMBIGUITY_SET)
if len(sigs) == 0:
    print('no signals this scan')
    print(f'  spot:     {SPOT.get("price")}')
    print(f'  event:    {TRACKED.get("event")}')
    print(f'  books:    {len(BOOKS)}')
else:
    cols = ['ticker', 'side', 'entry_price', 'model_p_yes', 'edge_c',
            'robust_pass_rate', 'robust_mean_edge_c', 'ttl_min']
    cols = [c for c in cols if c in sigs.columns]
    print(f'{len(sigs)} signal(s) passed all filters:\n')
    print(sigs[cols].to_string(index=False))

## 10. Open positions

In [ ]:
op = open_trades()
if len(op) == 0:
    print('no open positions')
else:
    cols = ['id', 'timestamp_utc', 'market_ticker', 'side', 'contracts',
            'entry_price', 'entry_edge_cents', 'model_p_yes', 'trade_type']
    cols = [c for c in cols if c in op.columns]
    print(f'{len(op)} open positions:\n')
    print(op[cols].to_string(index=False))

## 11. Robust filter decisions log

Every signal that came through `scan_signals` (pass or fail) is
logged here. Useful for tuning `robust_min_pass_rate` and
`robust_min_mean_edge_c`.

In [ ]:
from kalshi_v2.paper_db import _conn
import pandas as pd
conn = _conn()
rd = pd.read_sql_query(
    'SELECT ts, ticker, side, entry_price, n_measures, pass_rate, '
    'mean_edge_c, min_edge_c, max_edge_c, passed '
    'FROM robust_decisions ORDER BY ts DESC LIMIT 20', conn)
conn.close()
if len(rd) == 0:
    print('no robust decisions logged yet')
else:
    print(rd.to_string(index=False))

## 12. Portfolio metrics

Paper view (CFG bankroll) and live view (Kalshi balance) are kept
separate. Open positions get marked-to-market via in-memory WS
books with REST fallback.

In [ ]:
portfolio_metrics(kalshi_md=kalshi_md, kalshi_live=kalshi_live)

## 13. Recent settled trades

In [ ]:
st = settled_trades()
if len(st) == 0:
    print('no settled trades yet')
else:
    cols = ['id', 'market_ticker', 'side', 'contracts', 'entry_price',
            'settle_price', 'pnl_dollars', 'exit_reason', 'trade_type']
    cols = [c for c in cols if c in st.columns]
    print(f'{len(st)} settled trades, last 10:\n')
    print(st[cols].tail(10).to_string(index=False))
    print(f'\ntotal realized PnL: ${st["pnl_dollars"].sum():+.2f}')
    wins = (st['pnl_dollars'] > 0).sum()
    print(f'win rate: {wins}/{len(st)} = {wins/len(st)*100:.1f}%')

## 14. Risk-block log

Last 20 signals that were rejected by `risk_preflight`. Each entry
shows the reasons (ticker dedup, exposure cap, balance floor, etc.).

In [ ]:
if not RISK_BLOCKS:
    print('no risk blocks recorded')
else:
    for rb in RISK_BLOCKS[-20:]:
        print(f'  {rb["ts"][:19]}  {rb["ticker"]:30s} {rb["side"]:>3s}  '
              f'{rb["strategy"]}  → {"; ".join(rb["reasons"])}')

## 15. Controls

Run any of these as needed.

In [ ]:
# Stop the decision loop. Threads exit at next sleep wake (~250 ms).
# stop_bot()

# Hard kill: stop + force paper mode + refresh sessions.
# kill_switch()

# Flip mode to live. Refuses if balance unknown or $0.
# enable_live()

# Flip back to paper.
# disable_live()

# Cancel every resting Kalshi order.
# cancel_all_live_orders()